# Strategy 4 Evolution: Autonomous Data Agent

This notebook runs the refactored Strategy 4 pipeline with:
- Synthetic Traceback critic phase
- Lesson consolidation
- Semantic schema annotations
- Two-stage retrieval (vector + reranker)
- Global business-rule memory
- Memory governance (confidence/evidence + tentative TTL)
- Runtime-context-first retrieval (freshness-aware)
- Observability events to SQLite + optional Arize Phoenix
- Eval harness for baseline-vs-candidate regression checks


In [1]:
import os
from getpass import getpass

from datasets import load_dataset
from dotenv import load_dotenv

from agentic_sql import (
    ExperimentConfig,
    ModelConfig,
    build_runtime,
    require_openai_api_key,
    run_dual_memory_experiment,
)

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

require_openai_api_key()

dataset = load_dataset("xlangai/spider")
train_examples = dataset["train"]
test_examples = dataset["validation"]
print("Loaded Spider train examples:", len(train_examples))
print("Loaded Spider test/validation examples:", len(test_examples))

runtime_context_path = os.environ.get("RUNTIME_CONTEXT_PATH", "examples/runtime_context.sample.jsonl")
use_phoenix = os.environ.get("PHOENIX_ENABLED", "0") == "1"
phoenix_project = os.environ.get("PHOENIX_PROJECT_NAME", "agentic-sql")
phoenix_endpoint = os.environ.get("PHOENIX_COLLECTOR_ENDPOINT", "")
print("Runtime context path:", runtime_context_path)
print("Phoenix enabled:", use_phoenix)

model_cfg = ModelConfig(
    emb_provider="sentence_transformers",
    local_emb_model="all-MiniLM-L6-v2",
)
runtime = build_runtime(
    db_path="knowledge_base.db",
    model_cfg=model_cfg,
    bootstrap_from_kb=True,
    run_id=None,
)
llm = runtime.llm
schema_memory = runtime.schema_memory
reason_memory = runtime.reason_memory
global_memory = runtime.global_memory
kb_store = runtime.kb_store

print("SQLite KB path: knowledge_base.db")
print("Bootstrapped from SQLite:", runtime.bootstrap_stats)

# Keep a single fixed train backfill run so reruns upsert instead of append.
TRAIN_BACKFILL_RUN_ID = 900001
train_backfill_run_id = kb_store.backfill_step_events_from_examples(
    examples=train_examples,
    run_id=TRAIN_BACKFILL_RUN_ID,
    source_name="spider_train",
)
print("Train backfill run id:", train_backfill_run_id)
print("Step_events rows for train backfill run:", kb_store.count_step_events(run_id=train_backfill_run_id))
print("Total step_events rows (includes old historical runs):", kb_store.count_step_events())



Loaded Spider train examples: 7000
Loaded Spider test/validation examples: 1034
Runtime context path: examples/runtime_context.sample.jsonl
Phoenix enabled: False


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


SQLite KB path: knowledge_base.db
Bootstrapped from SQLite: {'run_id': 8, 'reason_lessons': 43, 'global_rules': 0, 'schema_annotations': 39}
Train backfill run id: 900001
Step_events rows for train backfill run: 7000
Total step_events rows (includes old historical runs): 7217


In [2]:
from agentic_sql import OfflineContextBuilder

builder = OfflineContextBuilder("knowledge_base.db")
human_annotations_path = "human_annotations.jsonl" if os.path.exists("human_annotations.jsonl") else None

pre_stats = builder.build_context(
    run_id=train_backfill_run_id,
    human_annotations_path=human_annotations_path,
    docs_dir="institutional_docs",
    code_dir=".",
    runtime_context_path=runtime_context_path if os.path.exists(runtime_context_path) else None,
)
embedded_count = builder.embed_context(embed_fn=llm.embed_texts, emb_model=llm.models.emb_model)
diag = builder.diagnostics_report(
    run_id=train_backfill_run_id,
    human_annotations_path=human_annotations_path,
    docs_dir="institutional_docs",
    code_dir=".",
    sample_limit=3,
)

print("Preprocess stats (ingested from train backfill):", pre_stats)
print("  table_usage_schema_annotations:", pre_stats.get("table_usage_schema_annotations", 0))
print("  table_usage_ast_crawl:", pre_stats.get("table_usage_ast_crawl", 0))
print("  qa_pairs:", pre_stats.get("qa_pairs", 0))
print("  runtime_context:", pre_stats.get("runtime_context", 0))
print("Embedded context items (new/updated):", embedded_count)
print("\nSource snapshot:", diag["source_snapshot"])
print("\nTotal context items by layer:", diag["total_counts_by_layer"])

print("\nLayer samples:")
for layer, samples in diag["layer_samples"].items():
    print(f"- {layer}:")
    for s in samples:
        print(f"    id={s['id']} title={s['title']} db_id={s['db_id']} scope={s['scope']}")

runtime_preview = builder.retrieve(
    question="is orders data stale or impacted by failures?",
    embed_fn=llm.embed_texts,
    top_k=5,
    db_id="sales_warehouse",
    layers=["runtime_context"],
    runtime_first_k=2,
    runtime_boost=0.15,
    runtime_max_age_hours=72,
)
print("\nRuntime context preview:")
for row in runtime_preview:
    print(f"- [{row['layer']}] {row['title']} (score={row['score']:.3f})")


Preprocess stats (ingested from train backfill): {'table_usage': 0, 'table_usage_schema_annotations': 0, 'table_usage_ast_crawl': 0, 'qa_pairs': 16, 'memory': 0, 'human_annotations': 0, 'codex_enrichment': 7, 'institutional_knowledge': 0, 'runtime_context': 5}
  table_usage_schema_annotations: 0
  table_usage_ast_crawl: 0
  qa_pairs: 16
  runtime_context: 5
Embedded context items (new/updated): 3

Source snapshot: {'latest_run_id': 900001, 'step_events_rows_total': 7218, 'step_events_rows_latest_run': 7000, 'reasoning_lessons_rows': 0, 'global_rules_rows': 0, 'schema_annotations_rows': 0, 'human_annotations_exists': False, 'institutional_docs_exists': False, 'institutional_doc_files': 0, 'code_dir_exists': True, 'candidate_code_files': 15}

Total context items by layer: {'codex_enrichment': 16, 'memory': 6992, 'runtime_context': 6, 'table_usage': 8189}

Layer samples:
- codex_enrichment:
    id=15392 title=pipeline_memory_ops.py db_id= scope=global
    id=15391 title=runtime.py db_id= 

In [ ]:
pre_reason_count = len(reason_memory.lessons)
pre_global_count = len(global_memory.lessons)
pre_schema_counts = {db_id: (len(m['tables']) + len(m['columns']) + len(m['join_edges'])) for db_id, m in schema_memory._store.items()}

cfg = ExperimentConfig(
    n_steps=100,
    initial_retrieval_k=20,
    retrieval_k=3,
    global_rule_k=3,
    context_top_k=10,
    context_per_layer_k=4,
    context_max_chars=5000,
    qa_top_k=6,
    qa_max_chars=1500,
    consolidation_k=6,
    consolidation_every_n=10,
    memory_ab_every_n=10,
    pass_sim_threshold=0.75,
    debug_trace=True,
    debug_context_chars=1200,
    min_global_confidence=0.8,
    min_global_evidence=2,
    tentative_ttl_steps=200,
    include_tentative_in_retrieval=False,
    runtime_context_first_k=2,
    runtime_context_boost=0.15,
    runtime_context_max_age_hours=48,
    planner_include_context_chars=1200,
    phoenix_enabled=use_phoenix,
    phoenix_project_name=phoenix_project,
    phoenix_endpoint=phoenix_endpoint,
)

stats, history = run_dual_memory_experiment(
    examples=test_examples,
    llm=llm,
    schema_memory=schema_memory,
    reason_memory=reason_memory,
    global_memory=global_memory,
    context_builder=builder,
    kb_store=kb_store,
    cfg=cfg,
    verbose=True,
)

print("Observability events in SQLite (all runs):", kb_store.count_observability_events())


[1] DEBUG INPUT
DB_ID: concert_singer
QUESTION:
How many singers do we have?

SCHEMA_HINT:
DB_ID=concert_singer
Known tables with descriptions:
- concert: Table containing data about concerts, including dates and other relevant details.
- singer: Table containing information about singers
- singer_in_concert: Seen in verified query.
- stadium: A venue for concerts and events, holding varying capacities.
Known join edges with descriptions:
- t1.concert_id = t2.concert_id: Verified join relationship.
- t1.concert_id = t3.concert_id: Verified join relationship.
- t1.singer_id = t2.singer_id: Verified join relationship.
- t1.stadium_id = t2.stadium_id: Join condition linking concert records to their respective stadiums.
Known columns with descriptions:
- age: Age of the singer
- average: Discovered from SQL AST (gold_ast_on_unknown_table).
- capacity: The maximum number of people that the stadium can accommodate.
- concert_id: Used in verified query.
- concert_name: Used in verified query.

In [ ]:
print("\nKey Metrics (on test/validation split):")
print("overall_pass_rate:", round(stats["overall_pass_rate"], 4))
print("first_try_accuracy:", round(stats["first_try_accuracy"], 4))
print("retry_recovery_rate:", round(stats["retry_recovery_rate"], 4))
print("exact_match_rate_first:", round(stats["exact_match_rate_first"], 4))
print("exact_match_rate_final:", round(stats["exact_match_rate_final"], 4))
print("avg_sim1:", round(stats["avg_sim1"], 4))
print("avg_sim2:", round(stats["avg_sim2"], 4))
print("sim_lift:", round(stats["sim_lift"], 4))
print("high_sim_non_exact_rate_final:", round(stats["high_sim_non_exact_rate_final"], 4))
print("schema_updates:", stats.get("schema_updates", 0))
print("tentative_lessons_stored:", stats.get("tentative_lessons_stored", 0))
print("expired_tentative_pruned:", stats.get("expired_tentative_pruned", 0))
print("planner_runs:", stats.get("planner_runs", 0))
print("ab_pass_rate_with_memory:", round(stats["ab_pass_rate_with_memory"], 4))
print("ab_pass_rate_without_memory:", round(stats["ab_pass_rate_without_memory"], 4))
print("llm_calls:", stats.get("llm_calls", 0))
print("llm_total_tokens:", stats.get("llm_total_tokens", 0))
print("llm_estimated_cost_usd:", round(stats.get("llm_estimated_cost_usd", 0.0), 6))
print("avg_llm_latency_ms:", round(stats.get("avg_llm_latency_ms", 0.0), 2))
print("phoenix_enabled:", stats.get("phoenix_enabled", False))
print("phoenix_setup_error:", stats.get("phoenix_setup_error", ""))

qa_counts = [int(r.get("related_qa_count", 0) or 0) for r in history]
print("avg_related_qa_count:", round(sum(qa_counts) / max(1, len(qa_counts)), 2))


In [ ]:
print("\nLearned During This Test Run")
print("=" * 80)

new_reason = list(zip(reason_memory.lessons[pre_reason_count:], reason_memory.meta[pre_reason_count:]))
new_global = list(zip(global_memory.lessons[pre_global_count:], global_memory.meta[pre_global_count:]))

print(f"New DB-specific lessons: {len(new_reason)}")
if new_reason:
    verified = sum(1 for _, m in new_reason if bool(m.get('verified', True)))
    tentative = len(new_reason) - verified
    print(f"  verified={verified}, tentative={tentative}")
    print("  sample lessons:")
    for i, (txt, meta) in enumerate(new_reason[:5], 1):
        one_line = txt.replace('\n', ' ')[:220]
        print(
            f"    {i}. db_id={meta.get('db_id')} status={meta.get('status')} verified={meta.get('verified', True)} "
            f"confidence={meta.get('confidence')} evidence={meta.get('evidence_count')} sim_before={meta.get('sim_before')} sim_after={meta.get('sim_after')}"
        )
        print(f"       {one_line}")

print(f"New global rules: {len(new_global)}")
if new_global:
    print("  sample global rules:")
    for i, (txt, meta) in enumerate(new_global[:5], 1):
        one_line = txt.replace('\n', ' ')[:220]
        print(
            f"    {i}. scope={meta.get('scope')} confidence={meta.get('confidence')} "
            f"evidence={meta.get('evidence_count')} sim_before={meta.get('sim_before')} sim_after={meta.get('sim_after')}"
        )
        print(f"       {one_line}")

planned = [r for r in history if r.get('planner_text')]
print(f"Examples with planner output: {len(planned)}")
if planned:
    print("  sample planner output:")
    for r in planned[:3]:
        ptxt = str(r.get('planner_text', '')).replace('\n', ' ')[:280]
        print(f"    idx={r.get('index')} db={r.get('db_id')} pass1={r.get('pass1')} pass2={r.get('pass2')}")
        print(f"       {ptxt}")

schema_growth = []
for db_id, m in schema_memory._store.items():
    now = len(m['tables']) + len(m['columns']) + len(m['join_edges'])
    before = pre_schema_counts.get(db_id, 0)
    if now > before:
        schema_growth.append((db_id, now - before, now, len(m['tables']), len(m['columns']), len(m['join_edges'])))

print(f"DBs with schema growth: {len(schema_growth)}")
if schema_growth:
    schema_growth.sort(key=lambda x: x[1], reverse=True)
    for db_id, delta, total, t, c, j in schema_growth[:10]:
        print(f"  {db_id}: +{delta} entries (total={total}; tables={t}, columns={c}, joins={j})")

learned_failures = [r for r in history if not r.get('pass2', r.get('pass1', False)) and r.get('lesson')]
print(f"Failed questions that still produced lessons: {len(learned_failures)}")
if learned_failures:
    print("  sample failure lessons:")
    for r in learned_failures[:5]:
        ltxt = str(r.get('lesson', '')).replace('\n', ' ')[:220]
        print(f"    idx={r.get('index')} db={r.get('db_id')} sim1={r.get('sim1')} sim2={r.get('sim2')}")
        print(f"       {ltxt}")
